# Inference

In [ ]:
%uv pip install torchcodec==0.7
%uv pip install nemo_toolkit['all']
%uv pip install datasets==4.4.1 
%uv pip install numpy==2.1.2
!pip install jiwer
%uv pip install pandas==2.3.2

# %sudo apt update
# %uv pip install nvidia-jetpack

Using Python 3.12.6 environment at: /usr/local
Resolved 1 package in 79ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
Prepared 1 package in 95ms
Installed 1 package in 97ms
 + torchcodec==0.7.0
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 340 packages in 9.48s
⠙ Preparing packages... (0/210)
⠙ Preparing packages... (0/210)
⠙ Preparing packages... (0/210)
pynacl     ------------------------------     0 B/1.34 MiB
⠙ Preparing

In [ ]:
import torch
import nemo.collections.asr as nemo_asr
from jiwer import wer, cer

from datasets import load_dataset

/usr/local/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[NeMo W 2025-12-20 19:35:44 nemo_logging:405] /usr/local/lib/python3.12/site-packages/lhotse/recipes/iwslt22_ta.py:323: SyntaxWarning: invalid escape sequence '\s'
      text = re.sub("\s+", " ", text)
    
[NeMo W 2025-12-20 19:35:44 nemo_logging:405] /usr/local/lib/python3.12/site-packages/lhotse/recipes/iwslt22_ta.py:324: SyntaxWarning: invalid escape sequence '\s'
      text = re.sub("\s+\.\s+", ".", text)
    
W1220 19:35:50.851000 502 site-packages/torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1220 19:35:50.851000 502 site-packages/torch/utils/cpp_extension.py:2425] If this is not desired

In [8]:
# Load dataset
ds = load_dataset("MohamedGomaa30/Egyptian-ASR-MGB-3", split="test")

In [9]:
ds

Dataset({
    features: ['audio', 'sentence'],
    num_rows: 200
})

In [10]:
ds[0]

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x2b0c6fe87110>,
 'sentence': 'وأين هي طرق التنمية وأين هي طرق التدريس التي تحفظ هذا الحق حق التعليم الفني يحتاج إلى كرامة أولاً ثم إلى طرق تدريس ثانياً وقبل هذا كله وبعده يحتاج الى المخلصون من هذا الوطن الذين يريدون حقاً لهذه القاطرة أن تنطلق'}

In [5]:
ds[0]['audio']['array']

array([-8.1787109e-03, -8.7280273e-03, -9.2468262e-03, ...,
        6.1035156e-05,  2.4414062e-04, -1.8310547e-04], dtype=float32)

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
from omegaconf import OmegaConf, open_dict
import torch
import nemo.collections.asr as nemo_asr

# 1. Load the model
model_name = "nvidia/stt_ar_fastconformer_hybrid_large_pcd_v1.0"
asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name)

# 2. Create a new decoding config with CUDA graphs explicitly disabled
# This ensures the greedy decoder doesn't try to use CUDA graphs
decoding_cfg = OmegaConf.create({
    'strategy': 'greedy_batch',
    'greedy': {
        'max_symbols_per_step': 10,
        'use_cuda_graph_decoder': False,  # Disable in greedy config
    },
    'use_cuda_graph_decoder': False,  # Disable at top level
})

# 3. Apply the decoding strategy
asr_model.change_decoding_strategy(decoding_cfg)

# 4. Additional safety: Disable CUDA graphs in the decoder itself if it exists
if hasattr(asr_model, 'decoding') and hasattr(asr_model.decoding, 'decoding'):
    decoder = asr_model.decoding.decoding
    if hasattr(decoder, 'decoding_computer'):
        # Force CUDA graphs mode to None
        decoder.decoding_computer.cuda_graphs_mode = None

# 5. Move to device and set to eval mode
device = "cuda" if torch.cuda.is_available() else "cpu"
asr_model.to(device)
asr_model.eval()



references = []
hypotheses = []

print("Starting transcription...")
for i, sample in enumerate(ds):
    if i >= 10: 
        break
    
    audio_array = sample["audio"]["array"]
    reference_text = sample["sentence"] # النص الأصلي
    
    if audio_array.ndim > 1:
        audio_array = np.mean(audio_array, axis=1)
        
    with torch.no_grad():
        try:
            transcriptions = asr_model.transcribe([audio_array])
            
            # في NeMo، النتيجة قد تكون قائمة نصوص أو كائن يحتوي على خاصية text
            if isinstance(transcriptions[0], str):
                result_text = transcriptions[0]
            else:
                result_text = transcriptions[0].text
            
            # تخزين النتائج للحساب الكلي
            references.append(reference_text)
            hypotheses.append(result_text)
            
            print(f"--- Sample {i+1} ---")
            print(f"Ref: {reference_text}")
            print(f"Hyp: {result_text}")
            
        except Exception as e:
            print(f"Error on sample {i+1}: {e}")
            continue

if len(references) > 0:
    total_wer = wer(references, hypotheses)
    total_cer = cer(references, hypotheses)

    print("\n" + "="*30)
    print(f"FINAL RESULTS (Samples: {len(references)})")
    print(f"Word Error Rate (WER): {total_wer:.4f}")
    print(f"Character Error Rate (CER): {total_cer:.4f}")
    print("="*30)

[NeMo W 2025-12-20 19:46:54 nemo_logging:405] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': True, 'num_workers': 8, 'pin_memory': True, 'max_duration': 20, 'min_duration': 0.5, 'is_tarred': True, 'tarred_audio_filepaths': '???', 'shuffle_n': 2048, 'bucketing_strategy': 'fully_randomized', 'bucketing_batch_size': None}
     Reason: Missing mandatory value: train_ds.manifest_filepath
        full_key: train_ds.manifest_filepath
        object_type=dict.
[NeMo W 2025-12-20 19:46:54 nemo_logging:405] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': False, 'use_start_end_token': False, 'num_workers': 8, 'pin_memory': True}
     Reason: Missing mandatory value: validation_ds.manifest_filepath
        full_key: validation_ds.manifest_filepath
        object_type=dict.
[NeMo W 2025-12-20 19:46:54 nemo_logging:405] Skipped conversion for conf

[NeMo I 2025-12-20 19:46:54 nemo_logging:393] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2025-12-20 19:46:55 nemo_logging:405] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': True, 'num_workers': 8, 'pin_memory': True, 'max_duration': 20, 'min_duration': 0.5, 'is_tarred': True, 'tarred_audio_filepaths': '???', 'shuffle_n': 2048, 'bucketing_strategy': 'fully_randomized', 'bucketing_batch_size': None}
     Reason: Missing mandatory value: train_ds.manifest_filepath
        full_key: train_ds.manifest_filepath
        object_type=dict.
[NeMo W 2025-12-20 19:46:55 nemo_logging:405] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': False, 'use_start_end_token': False, 'num_workers': 8, 'pin_memory': True}
     Reason: Missing mandatory value: validation_ds.manifest_filepath
        full_key: validation_ds.manifest_filepath
        object_type=dict.
[NeMo W 2025-12-20 19:46:55 nemo_logging:405] Skipped conversion for conf

[NeMo I 2025-12-20 19:46:56 nemo_logging:393] PADDING: 0
[NeMo I 2025-12-20 19:46:57 nemo_logging:393] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-12-20 19:46:57 nemo_logging:393] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-12-20 19:46:57 nemo_logging:393] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-12-20 19:46:58 nemo_logging:393] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--stt_ar_fastconformer_hybrid_large_pcd_v1.0/snapshots/7f32349d952f42a28dce979ba73270aa2bbdfa89/stt_ar_fastconformer_hybrid_large_pcd_v1.0.nemo.
[NeMo I 2025-12-20 19:46:58 nemo_logging:393] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-12-20 19:46:58 nemo_logging

Transcribing: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.32it/s]


--- Sample 1 ---
Reference:     وأين هي طرق التنمية وأين هي طرق التدريس التي تحفظ هذا الحق حق التعليم الفني يحتاج إلى كرامة أولاً ثم إلى طرق تدريس ثانياً وقبل هذا كله وبعده يحتاج الى المخلصون من هذا الوطن الذين يريدون حقاً لهذه القاطرة أن تنطلق
Transcription: التعليم وأين هي طرق التنمية وأين هي طرق التدريس التي تحفظ هذا الحق؟ حق التعليم الفني يحتاج إلى كرامة أولا ثم إلى طرق تدريس ثانيا وقبل هذا كله وبعده يحتاج إلى المخلصون من هذا الوطن الذين يريدون حقا لهذه القاطرة أن تنطلق
----------------------------------------


... (14 outputs left out)

Transcribing: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.13it/s]


--- Sample 9 ---
Reference:     بشويش أهو بالشكل ده كده هاخد بس عندي هنا في الصينية بتاعتي رشة كده من الدقيق أهي وهاخد عندي دي كده ادي دي كده أول واحدة طيب وهناخد كده التانية
Transcription: بس ايه بشويش اهو بالشكل ده كده هاخد بس عندي هنا فلطينية بتاعتي راشة كده من الدقيق اهي وهياخد عندي دي كده ادي كده اول واحدة طيب وهناخد كده الثانية
----------------------------------------


Transcribing: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.33it/s]


--- Sample 10 ---
Reference:     الكتابين الأولانين مش موجودين والكتاب التالت استعاره إبراهيم صالح شوفي في مكتبة الكلية ... متشكرة قوي ... العفو ... مع إبراهيم صالح دوري عليه في مكتبة الجامعة ... حاضر
Transcription: الكتابين الأولانيين مش موجودين الكتاب الثالث استعاره إبراهيم صالح تشوفي في مكتبته الكلية العب مع إبراهيم صالح تدوري عليك مكتبة الجنة
----------------------------------------
Transcription complete!
